In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementNotInteractableException, StaleElementReferenceException
from selenium.webdriver.support.ui import Select
import time
import pandas as pd

In [ ]:
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"

options = webdriver.ChromeOptions()
options.add_argument(f"user-agent={user_agent}")

service = Service("C:/Program Files (x86)/chromedriver-win64/chromedriver.exe")

driver = webdriver.Chrome(service=service, options=options)

driver.get("LINK") #add link
time.sleep(3)

In [ ]:
# open call for tenders, services

select = Select(driver.find_element("id", "ctl0_CONTENU_PAGE_AdvancedSearch_procedureType"))
select.select_by_value("1") 

select = Select(driver.find_element("id", "ctl0_CONTENU_PAGE_AdvancedSearch_categorie"))
select.select_by_value("3") 

driver.find_element("id", "ctl0_CONTENU_PAGE_AdvancedSearch_lancerRecherche").click() 

In [ ]:
# scraping

tenders =[]
while True:
    try:
        table = driver.find_element(By.XPATH, '//*[@id="tabNav"]/div[2]/div[2]/table')
        rows = table.find_elements(By.XPATH, ".//tbody/tr")

        for row in rows:
            td1 = row.find_element(By.CSS_SELECTOR, 'td[headers="cons_ref"] div:nth-of-type(4)')
            publish_date = td1.text.strip()

            
            td2 = row.find_element(By.CSS_SELECTOR, 'td[headers="cons_intitule"] div:nth-of-type(2)')
            title = td2.text.strip()
            if title.startswith("Objet :"):
                title = title.replace("Objet :", "", 1).strip()


            td3 = row.find_element(By.CSS_SELECTOR, 'td[headers="cons_dateEnd"]')
            lines = td3.text.strip().split("\n")
            expiration_date = lines[0] if lines else ""


            td4 = row.find_element(By.CSS_SELECTOR, 'td[headers="cons_actions"]')
            a= td4.find_element(By.TAG_NAME, 'a')
            link = a.get_attribute("href")
           
            tenders.append({
                "title": title,
                "publication_date": publish_date,
                "expiration_date": expiration_date,
                "link": link,
                "country": "Morocco"
            })
            

        try:
            next_button = driver.find_element(By.ID, "ctl0_CONTENU_PAGE_resultSearch_PagerTop_ctl2")
            if "aspNetDisabled" in next_button.get_attribute("class"):
                print("End of pagination.")
                break
            else:
                next_button.click()
                time.sleep(2)
        except (NoSuchElementException, ElementNotInteractableException):
            print("'Next' button not found or inactive.")
            break

    except StaleElementReferenceException:
        print("Stale element, attempting automatic reload...")
        time.sleep(2)
        continue  

print(tenders)

In [ ]:
# open call for tenders, supplies

driver.get("https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseAdvancedSearch&searchAnnCons")
time.sleep(3)

select = Select(driver.find_element("id", "ctl0_CONTENU_PAGE_AdvancedSearch_procedureType"))
select.select_by_value("1") 

select = Select(driver.find_element("id", "ctl0_CONTENU_PAGE_AdvancedSearch_categorie"))
select.select_by_value("2") 

driver.find_element("id", "ctl0_CONTENU_PAGE_AdvancedSearch_lancerRecherche").click() 

In [ ]:
# scraping

while True:
    try:
        table = driver.find_element(By.XPATH, '//*[@id="tabNav"]/div[2]/div[2]/table')
        rows = table.find_elements(By.XPATH, ".//tbody/tr")

        for row in rows:
            td1 = row.find_element(By.CSS_SELECTOR, 'td[headers="cons_ref"] div:nth-of-type(4)')
            publish_date = td1.text.strip()

            
            td2 = row.find_element(By.CSS_SELECTOR, 'td[headers="cons_intitule"] div:nth-of-type(2)')
            title = td2.text.strip()
            if title.startswith("Objet :"):
                title = title.replace("Objet :", "", 1).strip()


            td3 = row.find_element(By.CSS_SELECTOR, 'td[headers="cons_dateEnd"]')
            lines = td3.text.strip().split("\n")
            expiration_date = lines[0] if lines else ""


            td4 = row.find_element(By.CSS_SELECTOR, 'td[headers="cons_actions"]')
            a= td4.find_element(By.TAG_NAME, 'a')
            link = a.get_attribute("href")
           
            tenders.append({
                "title": title,
                "publication_date": publish_date,
                "expiration_date": expiration_date,
                "link": link,
                "country": "Morocco"
            })
            

        try:
            next_button = driver.find_element(By.ID, "ctl0_CONTENU_PAGE_resultSearch_PagerTop_ctl2")
            if "aspNetDisabled" in next_button.get_attribute("class"):
                print("End of pagination.")
                break
            else:
                next_button.click()
                time.sleep(2)
        except (NoSuchElementException, ElementNotInteractableException):
            print("'Next' button not found or inactive.")
            break

    except StaleElementReferenceException:
        print("Stale element, attempting automatic reload...")
        time.sleep(2)
        continue  

driver.quit()

print(tenders)

In [ ]:
df = pd.DataFrame(tenders)
df.to_excel("../2_scraped_files/Morocco.xlsx", index=False) # modify